# Day 3 — Tiny CIFAR-10 CNN

Follow the official PyTorch CIFAR-10 tutorial, keeping classification as a small learning vehicle.


## Load and normalize CIFAR-10

Source: this section follows the official PyTorch CIFAR-10 tutorial's first step: load and normalize the CIFAR-10 training and test datasets using `torchvision`.

`torchvision.transforms.v2` is the newer torchvision transform API. In this notebook it is imported as `v2`, so names like `v2.Compose` and `v2.Normalize` come from that API.

`v2.Compose([...])` is an ordered recipe. It sends each image through the listed transforms from top to bottom, so the output of one step becomes the input to the next step.

The transform pipeline is:

1. `v2.ToImage()` converts the loaded CIFAR-10 image into torchvision/PyTorch image format. The CNN needs tensor-like image data rather than a plain image object.
2. `v2.ToDtype(torch.float32, scale=True)` changes pixel values into floating-point numbers and scales them to about `0.0` through `1.0`. Neural networks train more predictably on small floating-point values than on raw integer pixel values.
3. `v2.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))` normalizes the red, green, and blue channels with `new_value = (old_value - mean) / std`. With mean `0.5` and standard deviation `0.5`, values near `0.0` become about `-1.0`, values near `0.5` become `0.0`, and values near `1.0` become about `1.0`.

We do these transformations because the model should see every image in a consistent numeric format, type, and scale. That makes the first training loop easier to reason about and gives the CNN a clean input signal from which to learn filters. Accuracy is only a sanity check here; the main reason to train this tiny classifier is to create learned feature maps we can inspect later.


In [3]:
import torch
import torchvision
from torchvision.transforms import v2

transform = v2.Compose(
    [
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
    ]
)

In [7]:
from pathlib import Path

import torchvision

current_dir: Path = Path.cwd()
PROJECT_ROOT: Path = next(
    (candidate for candidate in (current_dir, *current_dir.parents) if (candidate / "pyproject.toml").exists()),
    current_dir,
)
DATA_ROOT: Path = PROJECT_ROOT / "data"

print(f"Project root: {PROJECT_ROOT}")
print(f"CIFAR-10 data root: {DATA_ROOT}")

batch_size = 4
train_set = torchvision.datasets.CIFAR10(
    root=str(DATA_ROOT),
    train=True,
    download=True,
    transform=transform,
)

train_loader = torch.utils.data.DataLoader(
    train_set,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2,  # Matches the official tutorial; use 0 only if Windows multiprocessing errors occur.
)

test_set = torchvision.datasets.CIFAR10(
    root=str(DATA_ROOT),
    train=False,
    download=True,
    transform=transform,
)

test_loader = torch.utils.data.DataLoader(
    test_set,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
)

classes = (
    "plane",
    "car",
    "bird",
    "cat",
    "deer",
    "dog",
    "frog",
    "horse",
    "ship",
    "truck",
)

Project root: C:\Users\giloz\dev\cnn-feature-map-lab
CIFAR-10 data root: C:\Users\giloz\dev\cnn-feature-map-lab\data
